In [ ]:
# =============================================================================
# Fig2 (split):
#   Figure 2a — a: Flow maps 2020 vs 2060
#               b: 10x10 group sankey (East/West x 5 income tiers), 2020 | 2060
#   Figure 2b — a: Outbound patients by income group (East | West | Overall)
#               b: Weighted mean distance by income group (East | West | Overall)
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.patches import Rectangle, ConnectionPatch
from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm
from shapely.geometry import box as shapely_box
from pyproj import Transformer

# ── 0. Unified font sizes ─────────────────────────────────────────────────────
FS_BASE        = 7
FS_TITLE       = 8
FS_PANEL_LABEL = 10
FS_SCALE       = 6.5
FS_SANKEY      = 6.5   # node labels for the 10x10 sankey (kept consistent with FS_SCALE)

plt.rcParams.update({
    "font.family":      "Arial",
    "font.size":        FS_BASE,
    "axes.titlesize":   FS_TITLE,
    "axes.titleweight": "bold",
    "axes.titlepad":    3,
})

# ── 1. Paths ──────────────────────────────────────────────────────────────────
DATA_DIR   = Path("/Volumes/UCL/论文工作/空气污染/cross_flow_truncated/averaged_results/flow_avg") #/Volumes/UCL/论文工作/空气污染/cross_flow_by_disease/flow_results
DIST_PATH  = Path("/Volumes/UCL/论文工作/空气污染/city_distance/rail_distance_matrix.csv")
INCOME_DIR = Path("/Volumes/UCL/论文工作/空气污染/weighted_gdp/avg_fer_rcp/avg_2020.csv")
SHP_PATH   = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/city_shp/shi_en.shp")
CHINA_SHP  = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/中国底图-中图社审过版本/中国底图/中国面.shp")
CHINA_SHP2 = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/中国国界线/九段线/九段线和群岛.shp")

OUTFILE_A  = Path("/Users/shirley/Desktop/plots_V2/Fig2a_flowmap_sankey.png")  # NEW: flow map + sankey
OUTFILE_B  = Path("/Users/shirley/Desktop/plots_V2/Fig2b_dots.png")            # NEW: outbound count + distance

SCENARIO      = "earlypeak_NZ_CL"
PROJ_STR      = "+proj=aea +lat_1=25 +lat_2=47 +lat_0=0 +lon_0=105"
COMPARE_YEARS = [(2020, "a"), (2060, "b")]

# ── 2. Flow color scale ───────────────────────────────────────────────────────
FLOW_CMAP = plt.cm.plasma_r
FLOW_VMIN, FLOW_VMAX = 1, 2500
FLOW_NORM = mcolors.LogNorm(vmin=FLOW_VMIN, vmax=FLOW_VMAX)

# ── 3. Zoom regions ───────────────────────────────────────────────────────────
ZOOM_REGIONS = {
    "Beijing-Tianjin-Hebei": (113.5, 119.5, 36.0, 41.0),
    "Chengdu-Chongqing":     (102.5, 108.0, 28.0, 32.0),
    "Yangtze River Delta":   (118.5, 122.5, 29.5, 33.0),
    "Pearl River Delta":     (112.0, 115.5, 21.5, 24.5),
    "Urumqi":                (84.5,  90.5,  41.5, 46.0),
}
MAIN_MAP_BBOX = (80, 139, 10, 55)
ZOOM_ASPECT   = 1.4

# ── 4. Income & region settings ───────────────────────────────────────────────
INCOME_ORDER  = ["Low", "Lower middle", "Middle", "Upper middle", "High"]
INCOME_SHORT  = {
    "Low": "Low", "Lower middle": "L-Mid",
    "Middle": "Mid", "Upper middle": "U-Mid", "High": "High",
}
INCOME_COLORS = {
    "Low": "#afbeaa", "Lower middle": "#9dc497", "Middle": "#74c476",
    "Upper middle": "#31a354", "High": "#006d2c",
}
YEAR_COLORS   = {2020: "#4e79a7", 2060: "#e15759"}
# Sankey ribbon colors keyed by ORIGIN region — deliberately different from
# YEAR_COLORS so they're not confused with the 2020/2060 legend elsewhere.
REGION_SANKEY_COLORS = {"East": "#8c8c8c", "West": "#d98c3d"}
REGION_TITLES = ["East", "West", "Overall"]
CSV_CITY_FIELD = "city"

# ---- 10-group order for the sankey: West (Low->High) on top, East (Low->High) below ----
GROUP_ORDER = (
    [f"West-{INCOME_SHORT[g]}" for g in INCOME_ORDER] +
    [f"East-{INCOME_SHORT[g]}" for g in INCOME_ORDER]
)
GROUP_REGION = {g: ("West" if i < 5 else "East") for i, g in enumerate(GROUP_ORDER)}
GROUP_COLOR  = {g: INCOME_COLORS[INCOME_ORDER[i % 5]] for i, g in enumerate(GROUP_ORDER)}
GROUP_SHORT_LABEL = {
    g: f"{'W' if GROUP_REGION[g]=='West' else 'E'}-{g.split('-',1)[1]}"
    for g in GROUP_ORDER
}

# ── 5. Spatial data ───────────────────────────────────────────────────────────
china_border = gpd.read_file(CHINA_SHP).to_crs(PROJ_STR)
jiudanline   = gpd.read_file(CHINA_SHP2).to_crs(PROJ_STR)
city_shp     = gpd.read_file(SHP_PATH).to_crs(PROJ_STR)
city_shp["geometry"]  = city_shp.geometry.representative_point()
city_shp["cx"]        = city_shp.geometry.x
city_shp["cy"]        = city_shp.geometry.y
city_shp["city_name"] = city_shp["English"].str.strip()
city_pts = (city_shp.drop_duplicates(subset="city_name", keep="first")
            .set_index("city_name")[["cx", "cy"]].to_dict("index"))

city_shp_geo   = gpd.read_file(SHP_PATH)
cx_geo, cy_geo = city_shp_geo.geometry.centroid.x, city_shp_geo.geometry.centroid.y

def hhy_lon_at_lat(lat):
    t = (lat - 25.0) / (50.2 - 25.0)
    return 98.5 + t * (127.5 - 98.5)

city_shp_geo["region"] = np.where(cx_geo > hhy_lon_at_lat(cy_geo), "East", "West")
region_map = (city_shp_geo[["English", "region"]]
              .rename(columns={"English": "city"})
              .drop_duplicates(subset="city")
              .set_index("city")["region"].to_dict())

_tf = Transformer.from_crs("EPSG:4326", PROJ_STR, always_xy=True)
_HHY_X, _HHY_Y = _tf.transform([127.5, 98.5], [50.2, 25.0])
_NANHAI_BOUNDS = (gpd.GeoDataFrame(geometry=[shapely_box(105, 2, 122, 24)], crs="EPSG:4326")
                  .to_crs(PROJ_STR).total_bounds)

xmin, ymin, xmax, ymax = china_border.total_bounds
pad_x = (xmax - xmin) * 0.01
pad_y = (ymax - ymin) * 0.01

# ── 6. Geometry helpers ───────────────────────────────────────────────────────
def lonlat_bbox_to_proj(lon_min, lon_max, lat_min, lat_max):
    xs, ys = _tf.transform([lon_min, lon_max], [lat_min, lat_max])
    return min(xs), min(ys), max(xs), max(ys)

def pad_bbox_to_aspect(bbox, ratio):
    x0, y0, x1, y1 = bbox
    cx, cy = (x0+x1)/2, (y0+y1)/2
    w, h = x1-x0, y1-y0
    if w/h < ratio:
        x0, x1 = cx - h*ratio/2, cx + h*ratio/2
    else:
        y0, y1 = cy - w/ratio/2, cy + w/ratio/2
    return x0, y0, x1, y1

# ── 7. Matrix loaders ─────────────────────────────────────────────────────────
dist_raw = pd.read_csv(DIST_PATH, index_col=0)
dist_raw.index   = dist_raw.index.str.strip()
dist_raw.columns = dist_raw.columns.str.strip()

def load_matrix(year):
    path = DATA_DIR / f"flow_patientnum_avg_{SCENARIO}_{year}.csv"
    df   = pd.read_csv(path, index_col=0)
    df.index   = df.index.str.strip()
    df.columns = df.columns.str.strip()
    df = df[df.index.notna()]
    df = df.loc[~df.index.isin(["total"]), ~df.columns.isin(["total"])]
    np.fill_diagonal(df.values, 0)
    valid  = set(city_pts.keys())
    cities = [c for c in df.index if c in valid]
    return df.loc[cities, [c for c in df.columns if c in valid]]

def compute_all_edges(df):
    edges = []
    for ci in df.index:
        for cj in df.columns:
            if ci == cj: continue
            try:
                flow = float(df.loc[ci, cj])
            except KeyError:
                continue
            if np.isnan(flow) or flow <= 1: continue
            edges.append({"ori": ci, "dst": cj, "flow": flow})
    return pd.DataFrame(edges)

# ── 8. Income data ────────────────────────────────────────────────────────────
def load_income_map():
    df = pd.read_csv(INCOME_DIR).rename(columns={CSV_CITY_FIELD: "city"})
    df["city"] = df["city"].str.strip()
    return df.set_index("city")["income_group"].to_dict()

income_map = load_income_map()

def build_income_region_stats(year):
    """
    Returns:
      count_df : outbound patient-trips, index=INCOME_ORDER, cols=REGION_TITLES
      wdist_df : flow-weighted mean distance (km), same shape
    Based on origin city's income group and region.
    """
    df = load_matrix(year)
    records = []
    for ci in df.index:
        gi = income_map.get(ci)
        ri = region_map.get(ci, "East")
        if gi not in INCOME_ORDER: continue
        row = df.loc[ci]
        for cj in df.columns:
            if ci == cj: continue
            try:
                flow = float(row[cj])
                dist = float(dist_raw.loc[ci, cj])
            except (KeyError, ValueError):
                continue
            if np.isnan(flow) or flow <= 1: continue
            if np.isnan(dist) or dist <= 0: continue
            records.append({"income": gi, "region": ri, "flow": flow, "dist": dist})

    empty = pd.DataFrame(0.0, index=INCOME_ORDER, columns=REGION_TITLES)
    if not records:
        return empty, empty.copy()

    rec = pd.DataFrame(records)

    count = (rec.groupby(["income", "region"])["flow"].sum()
               .unstack(fill_value=0)
               .reindex(index=INCOME_ORDER, columns=["East", "West"], fill_value=0))
    count["Overall"] = count["East"] + count["West"]

    rec["fd"] = rec["flow"] * rec["dist"]
    def _wmean(sub):
        t = sub["flow"].sum()
        return sub["fd"].sum() / t if t > 0 else np.nan

    wdist = (rec.groupby(["income", "region"]).apply(_wmean)
               .unstack(fill_value=np.nan)
               .reindex(index=INCOME_ORDER, columns=["East", "West"]))
    wdist["Overall"] = rec.groupby("income").apply(_wmean).reindex(INCOME_ORDER)

    return count, wdist


def build_income_region_net_stats(year):
    """
    Net inflow (inflow - outflow) of intercity patient trips, aggregated by
    destination/origin city's income group and region.

    Positive = the group receives more patients than it sends out (net
    importer of care-seeking patients); negative = net exporter.

    Returns a DataFrame(index=INCOME_ORDER, cols=REGION_TITLES), same shape
    as build_income_region_stats's count_df, but values can be negative and
    (by construction) sum to ~0 across the whole country for a given year.
    """
    df = load_matrix(year)
    outflow_by_city = df.sum(axis=1)
    inflow_by_city  = df.sum(axis=0)
    net_by_city = inflow_by_city.reindex(df.index).fillna(0) - outflow_by_city

    records = []
    for city, net in net_by_city.items():
        income = income_map.get(city)
        region = region_map.get(city)
        if income not in INCOME_ORDER or region not in ("East", "West"):
            continue
        records.append({"income": income, "region": region, "net": net})

    empty = pd.DataFrame(0.0, index=INCOME_ORDER, columns=REGION_TITLES)
    if not records:
        return empty

    rec = pd.DataFrame(records)
    net_df = (rec.groupby(["income", "region"])["net"].sum()
                 .unstack(fill_value=0)
                 .reindex(index=INCOME_ORDER, columns=["East", "West"], fill_value=0))
    net_df["Overall"] = net_df["East"] + net_df["West"]
    return net_df


def _group_of(city):
    """Map a city -> one of the 10 GROUP_ORDER labels (e.g. 'West-Low'), or None."""
    region = region_map.get(city)
    income = income_map.get(city)
    if region is None or income not in INCOME_ORDER:
        return None
    return f"{region}-{INCOME_SHORT[income]}"


def merge_small_destinations(flow_mat, min_share=0.01):
    """
    Any destination GROUP whose AGGREGATE incoming share -- summed across
    ALL origins, as a fraction of the grand total -- falls below
    `min_share` is folded entirely into a single 'Other' destination
    column, and that group's own node/label is dropped from the right-hand
    side. Groups at or above the threshold keep their individual node.

    (Earlier version incorrectly thresholded each origin-destination cell
    against that origin's own row total, which could leave a destination
    with its own near-zero node even though its overall share was <1% --
    e.g. many origins each sending a "locally non-trivial" trickle to the
    same small destination. This version thresholds on the destination's
    true national share instead, so a destination is either fully kept or
    fully merged into 'Other' -- never left half-shown.)

    Returns a new DataFrame with the same index (GROUP_ORDER); columns are
    the destination groups at/above `min_share`, plus 'Other' placed FIRST
    (so it renders at the TOP of the right-side node stack).
    """
    grand_total = flow_mat.values.sum()
    col_totals = flow_mat.sum(axis=0)

    if grand_total <= 0:
        small_cols = list(flow_mat.columns)
        large_cols = []
    else:
        small_cols = [c for c in flow_mat.columns if (col_totals[c] / grand_total) < min_share]
        large_cols = [c for c in flow_mat.columns if c not in small_cols]

    mat = flow_mat[large_cols].copy()
    mat["Other"] = flow_mat[small_cols].sum(axis=1) if small_cols else 0.0

    mat = mat[["Other"] + large_cols]
    return mat


def build_group_flow_matrix(year):
    """
    Aggregate the city-level OD patient-flow matrix into a 10x10 group-level
    matrix: rows = origin group, cols = destination group, both in GROUP_ORDER.
    Same-group flows (including a city's flow to another city in its own
    group) are kept as self-loops (diagonal entries). No minimum-flow
    threshold is applied, per instructions: every nonzero city-pair edge
    (flow > 1) with both endpoints classifiable into a group is included.
    """
    df = load_matrix(year)
    mat = pd.DataFrame(0.0, index=GROUP_ORDER, columns=GROUP_ORDER)
    for ci in df.index:
        gi = _group_of(ci)
        if gi is None:
            continue
        row = df.loc[ci]
        for cj in df.columns:
            if ci == cj:
                continue
            try:
                flow = float(row[cj])
            except (KeyError, ValueError):
                continue
            if np.isnan(flow) or flow <= 1:
                continue
            gj = _group_of(cj)
            if gj is None:
                continue
            mat.loc[gi, gj] += flow
    return mat

# ── 9. Arrow drawing ──────────────────────────────────────────────────────────
def draw_arrows_on_ax(ax, edge_df, lw_scale=1.0, clip_bbox=None):
    if edge_df.empty: return
    plot_df = edge_df.copy()
    if clip_bbox is not None:
        x0, y0, x1, y1 = clip_bbox
        def _in(row):
            pt = city_pts.get(row["dst"])
            return pt is not None and x0<=pt["cx"]<=x1 and y0<=pt["cy"]<=y1
        plot_df = plot_df[plot_df.apply(_in, axis=1)]
        if plot_df.empty: return
    plot_df = plot_df.sort_values("flow")
    for _, row in plot_df.iterrows():
        try:
            ox=city_pts[row["ori"]]["cx"]; oy=city_pts[row["ori"]]["cy"]
            dx=city_pts[row["dst"]]["cx"]; dy=city_pts[row["dst"]]["cy"]
        except KeyError: continue
        flow  = row["flow"]
        color = FLOW_CMAP(FLOW_NORM(np.clip(flow, FLOW_VMIN, FLOW_VMAX)))
        t     = np.log10(np.clip(flow, FLOW_VMIN, FLOW_VMAX)) / np.log10(FLOW_VMAX)
        ax.add_patch(mpatches.FancyArrowPatch(
            (ox, oy), (dx, dy), connectionstyle="arc3,rad=0.15",
            arrowstyle="-|>" if flow >= 10 else "-",
            mutation_scale=(3+5*t)*lw_scale,
            linewidth=(0.15+0.9*t)*lw_scale,
            color=color, alpha=0.12+0.75*t,
            shrinkA=0, shrinkB=1, zorder=3+t, clip_on=True))

# ── 10. Map functions ─────────────────────────────────────────────────────────
def _add_scale_bar(ax):
    ax.add_artist(AnchoredSizeBar(
        ax.transData, 500_000, "500 km", loc="upper left",
        pad=0.3, borderpad=0.5, sep=3, color="black", frameon=False,
        size_vertical=(ymax-ymin)*0.003,
        fontproperties=fm.FontProperties(size=FS_SCALE, weight="bold")))

def draw_flow_map(ax_main, year, tag, total_n=None):
    """
    total_n : if given, appended to the title as "{year} (n=X)" where X is
              the total number of intercity patient trips that year
              (sum of the full OD matrix, i.e. every nonzero city-pair edge
              counted once). Pass the grand total of load_matrix(year).
    """
    df      = load_matrix(year)
    edge_df = compute_all_edges(df)
    china_border.plot(ax=ax_main, facecolor="#eef1f4", edgecolor="black", linewidth=0.3)
    city_shp.plot(ax=ax_main, facecolor="none", edgecolor="#b8bfc7", linewidth=0.15, zorder=1)
    jiudanline.plot(ax=ax_main, facecolor="white", edgecolor="black", linewidth=0.25, alpha=0.4)
    draw_arrows_on_ax(ax_main, edge_df)
    ax_main.plot(_HHY_X, _HHY_Y, color="black", lw=0.6, linestyle="--", dashes=(4,3), zorder=5)
    if MAIN_MAP_BBOX is not None:
        bx0,by0,bx1,by1 = lonlat_bbox_to_proj(*MAIN_MAP_BBOX)
        ax_main.set_xlim(bx0,bx1); ax_main.set_ylim(by0,by1)
    else:
        ax_main.set_xlim(xmin-pad_x, xmax+pad_x)
        ax_main.set_ylim(ymin+(ymax-ymin)*0.05, ymax+pad_y)
    _x0,_x1 = ax_main.get_xlim(); _y0,_y1 = ax_main.get_ylim()
    ax_main.set_aspect("equal")
    ax_main.set_box_aspect((_y1-_y0)/(_x1-_x0))
    ax_main.set_axis_off()
    ax_main.text(-0.02, 1.01, tag, transform=ax_main.transAxes,
                 fontsize=FS_PANEL_LABEL, fontweight="bold", va="bottom")
    title_str = str(year)
    if total_n is not None:
        title_str += f"  (n={total_n:,.0f})"
    ax_main.set_title(title_str, fontsize=FS_TITLE, fontweight="bold", pad=4)
    _add_scale_bar(ax_main)
    return edge_df

def draw_zoom_panel(ax, edge_df, bbox_proj, title):
    x0,y0,x1,y1 = bbox_proj
    china_border.plot(ax=ax, facecolor="#eef1f4", edgecolor="black", linewidth=0.3)
    city_shp.plot(ax=ax, facecolor="none", edgecolor="#b8bfc7", linewidth=0.2, zorder=1)
    draw_arrows_on_ax(ax, edge_df, lw_scale=1.3, clip_bbox=bbox_proj)
    ax.set_xlim(x0,x1); ax.set_ylim(y0,y1)
    ax.set_aspect("equal")
    ax.set_box_aspect((y1-y0)/(x1-x0))
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_edgecolor("black"); sp.set_linewidth(0.8)
    ax.set_title(title, fontsize=6, fontweight="normal", pad=1, loc="center")

def draw_zoom_rect_and_connector(fig, ax_main, ax_zoom, bbox_proj):
    x0,y0,x1,y1 = bbox_proj
    ax_main.add_patch(Rectangle((x0,y0),x1-x0,y1-y0,
                                  fill=False,edgecolor="black",linewidth=0.8,zorder=8))
    fig.add_artist(ConnectionPatch(
        xyA=(x1,(y0+y1)/2), coordsA="data", axesA=ax_main,
        xyB=(0,0.5),         coordsB="axes fraction", axesB=ax_zoom,
        color="black", linewidth=0.5, zorder=1))

def add_nanhai_inset(ax_main, edge_df):
    x0,y0,x1,y1 = _NANHAI_BOUNDS
    axins = ax_main.inset_axes([0.1, 0.02, 0.22, 0.25])
    axins.set_facecolor("white")
    china_border.plot(ax=axins, facecolor="#eef1f4", edgecolor="black", linewidth=0.3)
    jiudanline.plot(ax=axins, edgecolor="black", linewidth=0.4)
    draw_arrows_on_ax(axins, edge_df, lw_scale=0.8, clip_bbox=(x0,y0,x1,y1))
    axins.set_xlim(x0,x1); axins.set_ylim(y0,y1)
    axins.set_xticks([]); axins.set_yticks([])
    for sp in axins.spines.values():
        sp.set_edgecolor("black"); sp.set_linewidth(0.6)

def add_flow_colorbar(cax):
    sm = plt.cm.ScalarMappable(cmap=FLOW_CMAP, norm=FLOW_NORM)
    sm.set_array([])
    cb = plt.colorbar(sm, cax=cax, orientation="horizontal")
    cb.set_label("Outflow patients", fontsize=FS_BASE, labelpad=3)
    cb.ax.tick_params(labelsize=FS_SCALE, length=2)
    cb.ax.xaxis.set_major_formatter(
        plt.matplotlib.ticker.FuncFormatter(lambda x, _: f"{x:g}"))
    cb.outline.set_visible(False)

# ── 11. Income × region connected dot plot ────────────────────────────────────
def draw_income_dots(axes_row, data_by_year, ylabel, panel_label, y_scale=1, legend=True):
    """
    Connected dot plot: one dot per (income_group, year), joined by a dashed
    grey line; pct-change annotation in red (increase) or black (decrease).

    axes_row     : [ax_East, ax_West, ax_Overall]
    data_by_year : {year: DataFrame(index=INCOME_ORDER, cols=REGION_TITLES)}
    """
    yr0, yr1 = [yr for yr, _ in COMPARE_YEARS]   # e.g. 2020, 2060
    x        = np.arange(len(INCOME_ORDER))

    all_vals = np.array([
        data_by_year[yr][col].reindex(INCOME_ORDER).fillna(0).values / y_scale
        for yr in (yr0, yr1) for col in REGION_TITLES
    ])
    ymin_val = 0
    ymax_val = np.nanmax(all_vals) * 1.35

    for j, (ax, col) in enumerate(zip(axes_row, REGION_TITLES)):
        v0 = data_by_year[yr0][col].reindex(INCOME_ORDER).fillna(0).values / y_scale
        v1 = data_by_year[yr1][col].reindex(INCOME_ORDER).fillna(0).values / y_scale

        for xi, (val0, val1) in enumerate(zip(v0, v1)):
            ax.plot([xi, xi], [val0, val1],
                    color="#aaaaaa", linewidth=1.2,
                    linestyle=(0, (3, 2)),
                    zorder=1)

            ax.scatter(xi, val0, s=10, color=YEAR_COLORS[yr0], zorder=3, linewidths=0)
            ax.scatter(xi, val1, s=10, color=YEAR_COLORS[yr1], zorder=3, linewidths=0)

            if val0 > 0:
                pct   = (val1 - val0) / val0 * 100
                label = f"{pct:+.1f}%"
                color = "#C0392B" if pct > 0 else "#2c2c2c"
                ymid  = (val0 + val1) / 2
                ax.text(xi + 0.13, ymid, label,
                        fontsize=FS_SCALE , color=color,
                        va="center", ha="left", zorder=4)

        ax.set_xticks(x)
        ax.set_xticklabels([INCOME_SHORT[g] for g in INCOME_ORDER],
                            fontsize=FS_BASE, rotation=30, ha="right")
        ax.set_ylim(ymin_val, ymax_val)
        ax.tick_params(axis="y", labelsize=FS_SCALE)
        ax.spines[["top", "right"]].set_visible(False)
        ax.yaxis.set_major_locator(plt.MaxNLocator(4))
        ax.grid(axis="y", lw=0.4, alpha=0.35, linestyle="--", zorder=0)
        ax.text(0.04, 0.97, col, transform=ax.transAxes,
                fontsize=FS_BASE, fontstyle="italic",
                va="top", ha="left", color="#555555")

        if j == 0:
            ax.set_ylabel(ylabel, fontsize=FS_BASE)
        else:
            ax.set_yticklabels([])

    if legend:
        axes_row[-1].legend(
            handles=[
                plt.Line2D([0], [0], marker="o", color="w",
                           markerfacecolor=YEAR_COLORS[yr], markersize=4,
                           label=str(yr))
                for yr in (yr0, yr1)
            ],
            fontsize=FS_SCALE, frameon=False, loc="upper right"
        )

def draw_net_inflow_dots(axes_row, data_by_year, ylabel, y_scale=1_000, legend=True):
    """
    Connected dot plot for NET inflow (inflow - outflow), which can be
    negative. Unlike draw_income_dots, this:
      - allows the y-axis to extend below zero, with a solid zero line
      - annotates the ABSOLUTE change (delta), not percent change, since
        percent change is not meaningful when a group's net inflow crosses
        zero (flips from net exporter to net importer or vice versa)

    axes_row     : [ax_East, ax_West, ax_Overall]
    data_by_year : {year: DataFrame(index=INCOME_ORDER, cols=REGION_TITLES)}
    """
    yr0, yr1 = [yr for yr, _ in COMPARE_YEARS]
    x = np.arange(len(INCOME_ORDER))

    all_vals = np.array([
        data_by_year[yr][col].reindex(INCOME_ORDER).fillna(0).values / y_scale
        for yr in (yr0, yr1) for col in REGION_TITLES
    ])
    vmax = np.nanmax(np.abs(all_vals)) * 1.35 if np.nanmax(np.abs(all_vals)) > 0 else 1
    ymin_val, ymax_val = -vmax, vmax

    for j, (ax, col) in enumerate(zip(axes_row, REGION_TITLES)):
        v0 = data_by_year[yr0][col].reindex(INCOME_ORDER).fillna(0).values / y_scale
        v1 = data_by_year[yr1][col].reindex(INCOME_ORDER).fillna(0).values / y_scale

        ax.axhline(0, color="#999999", lw=0.6, zorder=0)

        for xi, (val0, val1) in enumerate(zip(v0, v1)):
            ax.plot([xi, xi], [val0, val1],
                    color="#aaaaaa", linewidth=1.2,
                    linestyle=(0, (3, 2)), zorder=1)

            ax.scatter(xi, val0, s=10, color=YEAR_COLORS[yr0], zorder=3, linewidths=0)
            ax.scatter(xi, val1, s=10, color=YEAR_COLORS[yr1], zorder=3, linewidths=0)

            delta = val1 - val0
            label = f"{delta:+.1f}"
            color = "#C0392B" if delta > 0 else "#2c2c2c"
            ymid  = (val0 + val1) / 2
            ax.text(xi + 0.13, ymid, label,
                    fontsize=FS_SCALE, color=color,
                    va="center", ha="left", zorder=4)

        ax.set_xticks(x)
        ax.set_xticklabels([INCOME_SHORT[g] for g in INCOME_ORDER],
                            fontsize=FS_BASE, rotation=30, ha="right")
        ax.set_ylim(ymin_val, ymax_val)
        ax.tick_params(axis="y", labelsize=FS_SCALE)
        ax.spines[["top", "right"]].set_visible(False)
        ax.yaxis.set_major_locator(plt.MaxNLocator(5))
        ax.grid(axis="y", lw=0.4, alpha=0.35, linestyle="--", zorder=0)
        ax.text(0.04, 0.97, col, transform=ax.transAxes,
                fontsize=FS_BASE, fontstyle="italic",
                va="top", ha="left", color="#555555")

        if j == 0:
            ax.set_ylabel(ylabel, fontsize=FS_BASE)
        else:
            ax.set_yticklabels([])

    if legend:
        axes_row[-1].legend(
            handles=[
                plt.Line2D([0], [0], marker="o", color="w",
                           markerfacecolor=YEAR_COLORS[yr], markersize=4,
                           label=str(yr))
                for yr in (yr0, yr1)
            ],
            fontsize=FS_SCALE, frameon=False, loc="upper right"
        )


# --------------------------------------------------------------------- #
# NEW: 10x10 group-level sankey (East/West x 5 income tiers)
#
# CAVEAT (please read): hand-built with bezier ribbons, not a dedicated
# sankey library. With gap>0 between stacked nodes, per-side height
# conservation is a close approximation, not pixel-perfect (see notes in
# earlier versions of this script). No minimum-flow threshold is applied
# here, per your instruction, so every nonzero origin-destination group
# pair is drawn -- expect a visually dense set of ribbons, especially for
# the large self-group (diagonal) flows.
# --------------------------------------------------------------------- #
def draw_group_sankey(ax, flow_mat, node_w=0.03, gap=0.0, title=None):
    """
    flow_mat : DataFrame(index=GROUP_ORDER, columns=[...]) of patient flow,
               origin (rows) -> destination (columns). Columns are usually
               GROUP_ORDER, but may include an extra 'Other' column produced
               by merge_small_destinations() to bundle low-share destinations.

    Left nodes  = origin groups (always GROUP_ORDER, 10 nodes), West Low->High
                  (top) then East Low->High (bottom).
    Right nodes = flow_mat.columns, in the given order (GROUP_ORDER, plus
                  'Other' appended at the bottom if present).
    Node color  = income tier for real groups (GROUP_COLOR); 'Other' nodes
                  use a neutral grey. Ribbon color = origin region
                  (REGION_SANKEY_COLORS).
    """
    left_names  = list(flow_mat.index)
    right_names = list(flow_mat.columns)
    grand_total = flow_mat.values.sum()
    if grand_total <= 0:
        ax.set_axis_off()
        return

    def _node_color(nm):
        return GROUP_COLOR.get(nm, "#bbbbbb")   # grey fallback for 'Other'

    def _node_label(nm):
        return GROUP_SHORT_LABEL.get(nm, nm)    # 'Other' shown as-is

    def _layout(totals, names):
        n = len(names)
        avail = 1.0 - gap * (n - 1)
        y_top = 1.0
        pos = {}
        for nm in names:
            h = max(totals[nm], 0) / grand_total * avail
            pos[nm] = [y_top - h, y_top]
            y_top -= (h + gap)
        return pos

    row_totals = flow_mat.sum(axis=1)   # outgoing, per origin group
    col_totals = flow_mat.sum(axis=0)   # incoming, per destination group/Other

    left_pos  = _layout(row_totals, left_names)
    right_pos = _layout(col_totals, right_names)

    x_left, x_right = 0.0, 1.0
    left_cursor  = {nm: left_pos[nm][1]  for nm in left_names}
    right_cursor = {nm: right_pos[nm][1] for nm in right_names}

    def _bezier_band(x0, y0_top, y0_bot, x1, y1_top, y1_bot, color, alpha):
        xm = x0 + (x1 - x0) * 0.5
        verts = [
            (x0, y0_top), (xm, y0_top), (xm, y1_top), (x1, y1_top),
            (x1, y1_bot), (xm, y1_bot), (xm, y0_bot), (x0, y0_bot),
            (x0, y0_top),
        ]
        codes = [MplPath.MOVETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4,
                 MplPath.LINETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4,
                 MplPath.CLOSEPOLY]
        ax.add_patch(PathPatch(MplPath(verts, codes), facecolor=color,
                                edgecolor="none", alpha=alpha, lw=0))

    for gi in left_names:
        for gj in right_names:
            flow = flow_mat.loc[gi, gj]
            if flow <= 0:
                continue
            h = flow / grand_total
            l_top = left_cursor[gi];  l_bot = l_top - h
            left_cursor[gi] = l_bot
            r_top = right_cursor[gj]; r_bot = r_top - h
            right_cursor[gj] = r_bot
            is_self = (gi == gj)
            _bezier_band(x_left, l_top, l_bot, x_right, r_top, r_bot,
                         color=REGION_SANKEY_COLORS[GROUP_REGION[gi]],
                         alpha=0.55 if not is_self else 0.75)

    for nm in left_names:
        y0, y1 = left_pos[nm]
        ax.add_patch(Rectangle((x_left - node_w, y0), node_w, y1 - y0,
                                facecolor=_node_color(nm), edgecolor="white", linewidth=0.5))
        pct = row_totals[nm] / grand_total * 100
        ax.text(x_left - node_w - 0.015, (y0 + y1) / 2,
                f"{_node_label(nm)} ({pct:.0f}%)",
                fontsize=FS_SANKEY, va="center", ha="right")

    for nm in right_names:
        y0, y1 = right_pos[nm]
        ax.add_patch(Rectangle((x_right, y0), node_w, y1 - y0,
                                facecolor=_node_color(nm), edgecolor="white", linewidth=0.5))
        pct = col_totals[nm] / grand_total * 100
        ax.text(x_right + node_w + 0.015, (y0 + y1) / 2,
                f"{_node_label(nm)} ({pct:.0f}%)",
                fontsize=FS_SANKEY, va="center", ha="left")

    if title:
        ax.text(0.5, 1.03, title, transform=ax.transAxes,
                fontsize=FS_TITLE, fontweight="bold", ha="center", va="bottom")

    ax.set_xlim(-0.30, 1.30)
    ax.set_ylim(-0.02, 1.05)
    ax.set_axis_off()


# ── 12. Pre-compute data ──────────────────────────────────────────────────────
count_by_year, dist_by_year = {}, {}
net_by_year = {}
for yr, _ in COMPARE_YEARS:
    count_by_year[yr], dist_by_year[yr] = build_income_region_stats(yr)
    net_by_year[yr] = build_income_region_net_stats(yr)

group_flow_by_year = {
    yr: merge_small_destinations(build_group_flow_matrix(yr), min_share=0.02)
    for yr, _ in COMPARE_YEARS
}

# ── 13a. FIGURE 2a: flow maps + two sankeys (2020 | 2060) ────────────────────
NZ  = len(ZOOM_REGIONS)
L_MAP    = 0.01
PL_X     = -0.005
PL_Y     = 0.98

fig_a = plt.figure(figsize=(180/25.4, 190/25.4), dpi=300)
subfigs_a = fig_a.subfigures(2, 1, height_ratios=[1, 1], hspace=0.06)

gs_left = GridSpec(
    NZ+2, 2, figure=subfigs_a[0],
    width_ratios=[3.4, 1],
    height_ratios=[0.3] + [1]*NZ + [0.3],
    left=L_MAP, right=0.47,
    top=0.95, bottom=0.10,
    hspace=0.32, wspace=0.2,
)
gs_right = GridSpec(
    NZ+2, 2, figure=subfigs_a[0],
    width_ratios=[3.4, 1],
    height_ratios=[0.3] + [1]*NZ + [0.3],
    left=0.53, right=1-L_MAP,
    top=0.95, bottom=0.10,
    hspace=0.32, wspace=0.2,
)

for block, (year, _) in enumerate(COMPARE_YEARS):
    gs = gs_left if block == 0 else gs_right
    ax_main  = subfigs_a[0].add_subplot(gs[:, 0])
    ax_zooms = [subfigs_a[0].add_subplot(gs[i+1, 1]) for i in range(NZ)]
    total_n  = load_matrix(year).values.sum()   # total intercity patient trips that year
    edge_df  = draw_flow_map(ax_main, year, tag="", total_n=total_n)
    add_nanhai_inset(ax_main, edge_df)
    for ax_z, (region_name, lonlat) in zip(ax_zooms, ZOOM_REGIONS.items()):
        raw_bbox  = lonlat_bbox_to_proj(*lonlat)
        bbox_proj = pad_bbox_to_aspect(raw_bbox, ZOOM_ASPECT)
        draw_zoom_panel(ax_z, edge_df, bbox_proj, region_name)
        draw_zoom_rect_and_connector(subfigs_a[0], ax_main, ax_z, bbox_proj)

subfigs_a[0].text(PL_X, PL_Y, "a", fontsize=FS_PANEL_LABEL,
                   fontweight="bold", va="top", ha="left")

cax = subfigs_a[0].add_axes([0.13, 0.1, 1 - 2*0.13, 0.02])
add_flow_colorbar(cax)

# two sankeys side by side, 2020 | 2060
gs_sankey = GridSpec(1, 2, figure=subfigs_a[1],
                      left=0.08, right=0.97, top=0.90, bottom=0.05,
                      wspace=0.55)
for k, (year, _) in enumerate(COMPARE_YEARS):
    ax_sk = subfigs_a[1].add_subplot(gs_sankey[0, k])
    draw_group_sankey(ax_sk, group_flow_by_year[year], title=str(year))

subfigs_a[1].text(PL_X, 1.0, "b", fontsize=FS_PANEL_LABEL,
                   fontweight="bold", va="top", ha="left")

OUTFILE_A.parent.mkdir(parents=True, exist_ok=True)
fig_a.savefig(OUTFILE_A, dpi=400, bbox_inches="tight", facecolor="white")
plt.close(fig_a)
print(f"✓ Saved → {OUTFILE_A}")


# ── 13b. FIGURE 2b: outbound-patient count + distance dot plots ──────────────
L_PANEL = 0.08

fig_b = plt.figure(figsize=(180/25.4, 92/25.4), dpi=300)
subfigs_b = fig_b.subfigures(2, 1, height_ratios=[1, 1], hspace=0.25)

gs_top = GridSpec(1, 3, figure=subfigs_b[0],
                   left=L_PANEL, right=0.97,
                   top=0.90, bottom=0.15,
                   wspace=0.18)
axes_top = [subfigs_b[0].add_subplot(gs_top[0, j]) for j in range(3)]
draw_net_inflow_dots(axes_top, net_by_year,
                      ylabel="Net inflow patients (x10³)", y_scale=1_000,
                      legend=False)
subfigs_b[0].text(PL_X, 1.02, "a", fontsize=FS_PANEL_LABEL,
                   fontweight="bold", va="top", ha="left")

gs_bot = GridSpec(1, 3, figure=subfigs_b[1],
                   left=L_PANEL, right=0.97,
                   top=0.90, bottom=0.15,
                   wspace=0.18)
axes_bot = [subfigs_b[1].add_subplot(gs_bot[0, j]) for j in range(3)]
draw_income_dots(axes_bot, dist_by_year,
                 ylabel="Mean travel distance (km)", panel_label="b", y_scale=1,
                 legend=False)
subfigs_b[1].text(PL_X, 1.02, "b", fontsize=FS_PANEL_LABEL,
                   fontweight="bold", va="top", ha="left")

# single shared legend for the whole figure (2020/2060 apply identically to
# both rows), placed in the top-right margin, outside any panel's data area
# so it never overlaps the dots or the delta/pct annotations.
fig_b.legend(
    handles=[
        plt.Line2D([0], [0], marker="o", color="w",
                   markerfacecolor=YEAR_COLORS[yr], markersize=5,
                   label=str(yr))
        for yr, _ in COMPARE_YEARS
    ],
    fontsize=FS_BASE, frameon=False, ncol=2,
    loc="upper right", bbox_to_anchor=(0.99, 1.02),
)

OUTFILE_B.parent.mkdir(parents=True, exist_ok=True)
fig_b.savefig(OUTFILE_B, dpi=400, bbox_inches="tight", facecolor="white")
plt.close(fig_b)
print(f"✓ Saved → {OUTFILE_B}")

/var/folders/xt/zxpvbvsx2zb75gd2395zm9000000gn/T/ipykernel_62348/858175310.py:112: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cx_geo, cy_geo = city_shp_geo.geometry.centroid.x, city_shp_geo.geometry.centroid.y
/var/folders/xt/zxpvbvsx2zb75gd2395zm9000000gn/T/ipykernel_62348/858175310.py:227: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  wdist = (rec.groupby(["income", "region"]).apply(_wmean)
/var/folders/xt/zxpvbvsx2zb75gd2395zm9000000gn/T/ipykernel_62348/858175310.py:230: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior i

✓ Saved → /Users/shirley/Desktop/plots_V2/Fig2a_flowmap_sankey.png
✓ Saved → /Users/shirley/Desktop/plots_V2/Fig2b_dots.png
